## Example on how to evaluate a vision encoder with the Hummingbird or Dense NN Retrieval Evaluation

<a href="https://githubtocolab.com/vpariza/open-hummingbird-eval/blob/main/examples/hbird_eval_example_faiss_gpu.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

### 1. Install required Libraries

In [ ]:
# ---- PyTorch (CUDA 12.1) ----
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121
!pip install lightning==2.4.0
!pip install torchmetrics==1.7.0
!pip install tqdm==4.67.1  # for progress bars
!pip install scipy==1.15.2
!pip install joblib==1.4.2
!pip install numpy==1.26.4  # not installed; torch bundles correct triton
# !pip install faiss-gpu-cu12  # not used; CUDA-specific wheel may mismatch Colab runtime
# !pip install faiss-gpu  # not available for Python 3.12 in Colab
!pip install faiss-cpu  # using CPU FAISS since Colab runs Python 3.12; original experiments used Python 3.11 with faiss-gpu-cu12
# !pip uninstall -y thinc  # unnecessary in fresh Colab runtime
!pip install -q gdown  # for MVImgNet subset
!pip install pyyaml  # required for ModelCheckpoint in pytorch_lightning
!pip install transformers>=4.34.0 huggingface-hub>=0.16.4  # needed for HF models



Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.1 MB/s eta 0:00:00


**Important**: After the installation step please restart the Runtime/Kernel before continuing with the step 2.

### 2. Access MVImgNet subset and clone our repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/datasets/mvimgnet"
# ToDo that won't work for other people!

100  113  125  126  152  166  19  196  46  57  60  7  70  8  99


In [ ]:
!git clone https://github.com/ToyeshC/open-hummingbird-3d-eval.git

Cloning into 'open-hummingbird-3d-eval'...
remote: Enumerating objects: 1075, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 1075 (delta 26), reused 15 (delta 15), pack-reused 1032 (from 1)
Receiving objects: 100% (1075/1075), 110.31 MiB | 19.79 MiB/s, done.
Resolving deltas: 100% (665/665), done.


In [ ]:
# Move to the repository folder
%cd open-hummingbird-3d-eval

# Checkout to branch with multiview evaluation
!git checkout add-3d-evaluation

/content/open-hummingbird-3d-eval
Branch 'add-3d-evaluation' set up to track remote branch 'add-3d-evaluation' from 'origin'.
Switched to a new branch 'add-3d-evaluation'


### 3. Unzip Contents of zip Dataset

In [ ]:
# !unzip voc_data.zip

### 4. Install repo

In [ ]:
!pip install .

Processing /content/open-hummingbird-3d-eval
  Preparing metadata (setup.py) ... done
  Created wheel for open-hummingbird-eval: filename=open_hummingbird_eval-1.0.0-py3-none-any.whl size=24509 sha256=ef069e807dce9b30f4cc595095708608927dadc8c682ca059ed494d2a67dbbc6
  Stored in directory: /root/.cache/pip/wheels/c3/56/43/32c832e60200bccbf288e909aa5eba444898ad008c625d6387
Successfully built open-hummingbird-eval
  Attempting uninstall: open-hummingbird-eval
    Found existing installation: open-hummingbird-eval 1.0.0
    Uninstalling open-hummingbird-eval-1.0.0:
      Successfully uninstalled open-hummingbird-eval-1.0.0


In [ ]:
# ToDo: comma missing

# %%writefile setup.py
# from setuptools import setup, find_packages

# setup(
#     name="open-hummingbird-eval",
#     version="1.0.0",
#     author="Valentinos Pariza, Mohammadreza Salehi, Yuki Asano",
#     author_email="valentinos.pariza@utn.de",
#     description="A library to evaluate the effectiveness of spatial features acquired from a vision encoder on a training dataset, to associate themselves to relevant features from a dataset (validation), through the utilization of a k-NN classifier/retriever.",
#     long_description=open("README.md").read(),
#     long_description_content_type="text/markdown",
#     url="",
#     packages=find_packages(),
#     install_requires=[
#         "scipy>=1.11.4,<2.0.0",
#         "numpy>=1.26.4,<2.0.0",
#         "tqdm~=4.67.1",
#         "lightning>=2.3.0",
#     ],
#     classifiers=[
#         "Programming Language :: Python :: 3",
#         "License :: OSI Approved :: MIT License",
#         "Operating System :: OS Independent",
#     ],
#     python_requires=">=3.7",
# )



### 5. Evaluate a preferred model on the downloaded dataset

In [ ]:
import torch
# from hbird.hbird_eval_3d import hbird_evaluation
from hbird.hbird_eval import hbird_evaluation

In [ ]:
# Parameters for the model dino
device = 'cuda'
input_size = 512  # 224
batch_size = 4  # 64
patch_size = 16
embed_dim = 768  # 384
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
num_workers = 8
n_neighbours = 30
nn_method = 'faiss'
memory_size = 1024000
augmentation_epoch = 1


Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_deitsmall16_pretrain.pth
100%|██████████| 82.7M/82.7M [00:00<00:00, 95.3MB/s]


In [ ]:
# Dataset Configurations
dataset_name = 'mvimgnet'
data_dir = './split_angles_mvimgnet'
# todo
train_fs_path = './split_angles_mvimgnet/sets/trainaug.txt'
val_fs_path = './split_angles_mvimgnet/sets/val.txt'

In [ ]:
def extract_dino_features(model, imgs):
    return model.get_intermediate_layers(imgs)[0][:, 1:], None

In [ ]:
hbird_miou = hbird_evaluation(model.to(device),
        d_model=embed_dim,        # size of the embedding feature vectors of patches
        patch_size=patch_size,
        batch_size = batch_size,
        input_size=input_size,
        augmentation_epoch=1,     # how many iterations of augmentations to use on top of the training dataset in order to generate the memory
        device=device,
        return_knn_details=False, # whether to return additional NNs details
        nn_method='faiss',
        n_neighbours=30,         # the number of neighbors to fetch per image patch
        nn_params=None,           # Other parameters to be used for the k-NN operator
        ftr_extr_fn=extract_dino_features,           # function that extracts features from a vision encoder on images
        dataset_name=dataset_name,       # the name of the dataset to use, currently only Pascal VOC is included.
        data_dir=data_dir,    # path to the dataset to use for evaluation
        memory_size=None,
        train_fs_path=train_fs_path,
        val_fs_path=val_fs_path)
print('Hummingbird Evaluation (mIoU):', hbird_miou)

FileNotFoundError: [Errno 2] No such file or directory: './split_angles_mvimgnet/sets/trainaug.txt'